# Monitoring & Observability for ML

Simulate drift, detect it with PSI / KS, and connect detector alerts to actual
model degradation.

## 1. Baseline dataset and a shifted (drifted) version

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 5000

def make_data(loc_x1=0.0, scale_x2=1.0, n=N):
    x1 = rng.normal(loc_x1, 1.0, n)
    x2 = rng.normal(2.0, scale_x2, n)
    # true relationship (the 'concept'): y depends on x1 + 0.5*x2
    y = (x1 + 0.5 * x2 + rng.normal(0, 0.5, n) > 1.0).astype(int)
    return pd.DataFrame({"x1": x1, "x2": x2, "y": y})

baseline = make_data()
shifted = make_data(loc_x1=1.2, scale_x2=1.8)  # x1 mean shifted, x2 variance inflated
baseline.describe().loc[["mean", "std"]]

## 2. PSI and KS between baseline and shifted distributions

In [ ]:
from scipy import stats

def psi(reference, current, bins=10):
    edges = np.histogram_bin_edges(reference, bins=bins)
    edges[0], edges[-1] = -np.inf, np.inf  # catch out-of-range production values
    ref_pct = np.histogram(reference, bins=edges)[0] / len(reference)
    cur_pct = np.histogram(current, bins=edges)[0] / len(current)
    ref_pct = np.clip(ref_pct, 1e-6, None)
    cur_pct = np.clip(cur_pct, 1e-6, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

for col in ["x1", "x2"]:
    p = psi(baseline[col], shifted[col])
    ks = stats.ks_2samp(baseline[col], shifted[col])
    label = "stable" if p < 0.1 else ("watch" if p < 0.25 else "ALERT")
    print(f"{col}: PSI={p:.3f} ({label})  KS stat={ks.statistic:.3f} p={ks.pvalue:.2e}")

## 3. Evidently drift report (optional — skips gracefully if not installed)

In [ ]:
try:
    from evidently import Report
    from evidently.presets import DataDriftPreset

    report = Report([DataDriftPreset()])
    snapshot = report.run(reference_data=baseline[["x1", "x2"]],
                          current_data=shifted[["x1", "x2"]])
    snapshot.save_html("evidently_drift_report.html")
    print("saved evidently_drift_report.html — open it in a browser")
except ImportError:
    print("evidently not installed — `pip install evidently` (API shown is for >=0.7)")

## 4. Simulate a production feed with gradually shifting inputs

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression().fit(baseline[["x1", "x2"]], baseline["y"])

WINDOWS, WINDOW_SIZE, PSI_THRESHOLD = 30, 500, 0.25
records = []
for w in range(WINDOWS):
    drift_amount = max(0.0, (w - 10) * 0.15)   # drift starts at window 10, grows
    window = make_data(loc_x1=drift_amount, n=WINDOW_SIZE)
    records.append({
        "window": w,
        "psi_x1": psi(baseline["x1"], window["x1"]),
        "accuracy": model.score(window[["x1", "x2"]], window["y"]),
    })

monitor = pd.DataFrame(records)
monitor["alert"] = monitor["psi_x1"] > PSI_THRESHOLD
monitor.head(15)

## 5. Mini-project: detector alerts vs. actual accuracy degradation

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.plot(monitor["window"], monitor["accuracy"], color="tab:blue", label="accuracy")
ax1.set_xlabel("window"); ax1.set_ylabel("accuracy", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(monitor["window"], monitor["psi_x1"], color="tab:orange", label="PSI(x1)")
ax2.axhline(PSI_THRESHOLD, color="tab:red", ls="--", lw=1, label="PSI threshold")
ax2.set_ylabel("PSI", color="tab:orange")

first_alert = monitor.loc[monitor["alert"], "window"].min()
if pd.notna(first_alert):
    ax1.axvline(first_alert, color="tab:red", alpha=0.4)
    ax1.set_title(f"First alert at window {int(first_alert)} — compare with when accuracy actually decays")
plt.tight_layout(); plt.show()

# Question to answer in a sentence below: did the input-drift alert fire BEFORE
# accuracy visibly degraded? That lead time is the value of input monitoring.